In [1]:
import os
import glob
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv
from datetime import datetime

In [2]:
# ── Load DB credentials from .env ──────────────────────────────────────────
load_dotenv()

DB_CONFIG = {
    "host":     os.getenv("DB_HOST",     "localhost"),
    "port":     int(os.getenv("DB_PORT", "5432")),
    "dbname":   os.getenv("DB_NAME",     "bvmt_db"),
    "user":     os.getenv("DB_USER",     "postgres"),
    "password": os.getenv("DB_PASSWORD", ""),
}

In [3]:
# ── Configuration ───────────────────────────────────────────────────────────
DATA_FOLDER  = r"C:\Users\Negza\Desktop\projects\pfe\bvmt_project\data\prices"
TARGET_GROUP = 11   # Main market stocks only (Groupe 11)



In [9]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 1: File Parsers
# ═══════════════════════════════════════════════════════════════════════════

def parse_csv(filepath: str) -> pd.DataFrame:
    """Parse semicolon-separated CSV files (2022-2025 format)."""
    df = pd.read_csv(filepath, sep=';', encoding='utf-8')
    df.columns = [c.strip() for c in df.columns]
    return df


def _normalize_numeric(series: pd.Series) -> pd.Series:
    """Convert locale-formatted numeric text to numeric values."""
    cleaned = series.astype(str).str.strip()
    cleaned = cleaned.str.replace('\u00A0', '', regex=False)
    cleaned = cleaned.str.replace(' ', '', regex=False)
    cleaned = cleaned.str.replace(',', '.', regex=False)
    cleaned = cleaned.replace({'': None, 'nan': None, 'None': None})
    return pd.to_numeric(cleaned, errors='coerce')


def parse_txt(filepath: str) -> pd.DataFrame:
    """Parse fixed-width TXT files (2016-2021 format)."""
    df = pd.read_fwf(filepath, skiprows=[1], encoding='utf-8')
    df.columns = [c.strip() for c in df.columns]

    # 2018/2019 files use NB_TRAN instead of NB_TRANSACTION.
    if 'NB_TRANSACTION' not in df.columns and 'NB_TRAN' in df.columns:
        df = df.rename(columns={'NB_TRAN': 'NB_TRANSACTION'})

    # Remove any repeated header rows or leftover separator lines.
    if 'SEANCE' in df.columns:
        df = df[~df['SEANCE'].astype(str).str.startswith('---')]
        df = df[df['SEANCE'] != 'SEANCE']
        df = df[df['SEANCE'].notna()]

    # Convert known numeric columns when they exist.
    numeric_columns = [
        'OUVERTURE',
        'CLOTURE',
        'PLUS_BAS',
        'PLUS_HAUT',
        'CAPITAUX',
        'QUANTITE_NEGOCIEE',
        'NB_TRANSACTION',
        'GROUPE',
    ]
    for col in numeric_columns:
        if col in df.columns:
            df[col] = _normalize_numeric(df[col])

    return df


def load_file(filepath: str) -> pd.DataFrame:
    """Auto-detect format by extension and parse accordingly."""
    ext = os.path.splitext(filepath)[1].lower()

    print(f"  📂 Reading: {os.path.basename(filepath)}")

    if ext == '.csv':
        df = parse_csv(filepath)
    elif ext == '.txt':
        df = parse_txt(filepath)
    else:
        print(f"  ⚠️  Unknown extension '{ext}' — skipping.")
        return pd.DataFrame()

    return df

In [7]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 2: Clean & Filter
# ═══════════════════════════════════════════════════════════════════════════

def clean_and_filter(df: pd.DataFrame, source_file: str) -> pd.DataFrame:
    """Keep only main market stocks and standardise all columns."""
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]

    # Accept known source-column aliases before validation.
    alias_map = {
        'NB_TRAN': 'NB_TRANSACTION',
        'C_GR_RLC': 'GROUPE',
        'CODE_VAL': 'CODE',
        'LIB_VAL': 'VALEUR',
    }
    for source_col, target_col in alias_map.items():
        if target_col not in df.columns and source_col in df.columns:
            df = df.rename(columns={source_col: target_col})

    required_columns = [
        'SEANCE', 'CODE', 'VALEUR', 'GROUPE', 'OUVERTURE', 'CLOTURE',
        'PLUS_BAS', 'PLUS_HAUT', 'QUANTITE_NEGOCIEE', 'NB_TRANSACTION', 'CAPITAUX'
    ]
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        print(f"  ❌ Missing required columns in {os.path.basename(source_file)}: {missing_columns}")
        return pd.DataFrame()

    # Strip whitespace from string columns.
    for col in ['SEANCE', 'CODE', 'VALEUR']:
        df[col] = df[col].astype(str).str.strip()

    # Normalize numeric source columns before filtering.
    for col in ['GROUPE', 'OUVERTURE', 'CLOTURE', 'PLUS_BAS', 'PLUS_HAUT',
                'QUANTITE_NEGOCIEE', 'NB_TRANSACTION', 'CAPITAUX']:
        df[col] = _normalize_numeric(df[col])

    # Filter: main market only (GROUPE == 11).
    df = df[df['GROUPE'] == TARGET_GROUP].copy()

    if df.empty:
        print(f"  ⚠️  No GROUPE=11 rows found in {source_file}")
        return df

    # Parse date, supporting both 4-digit and 2-digit years.
    seance_text = df['SEANCE'].astype(str).str.strip()
    parsed_dates = pd.to_datetime(seance_text, format='%d/%m/%Y', errors='coerce')
    missing_date_mask = parsed_dates.isna()
    if missing_date_mask.any():
        parsed_dates.loc[missing_date_mask] = pd.to_datetime(
            seance_text.loc[missing_date_mask],
            format='%d/%m/%y',
            errors='coerce',
        )
    df['seance'] = parsed_dates.dt.date

    # Drop rows where date couldn't be parsed.
    bad_dates = df['seance'].isna().sum()
    if bad_dates > 0:
        print(f"  ⚠️  Dropped {bad_dates} rows with unparseable dates")
        df = df[df['seance'].notna()]

    # Rename columns to match our database schema.
    df = df.rename(columns={
        'CODE':               'isin_code',
        'VALEUR':             'ticker',
        'OUVERTURE':          'ouverture',
        'CLOTURE':            'cloture',
        'PLUS_BAS':           'plus_bas',
        'PLUS_HAUT':          'plus_haut',
        'QUANTITE_NEGOCIEE':  'quantite_negociee',
        'NB_TRANSACTION':     'nb_transaction',
        'CAPITAUX':           'capitaux',
    })

    # Keep only the columns we need.
    keep = ['seance', 'isin_code', 'ticker', 'ouverture', 'cloture',
            'plus_bas', 'plus_haut', 'quantite_negociee', 'nb_transaction', 'capitaux']
    df = df[keep]

    print(f"  ✅ {len(df):,} rows | {df['ticker'].nunique()} stocks | "
          f"{df['seance'].min()} → {df['seance'].max()}")

    return df



In [14]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 3: Database Operations
# ═══════════════════════════════════════════════════════════════════════════

def get_connection():
    """Create and return a psycopg2 connection."""
    return psycopg2.connect(**DB_CONFIG)


def upsert_company_metadata(conn, df: pd.DataFrame):
    """
    Insert or update company_metadata for every unique stock in df.
    Keeps the company row stable across yearly files.
    """
    companies = df.groupby('isin_code').agg(
        ticker          = ('ticker', 'first'),
        first_seen_date = ('seance', 'min'),
        last_seen_date  = ('seance', 'max'),
    ).reset_index()

    sql = """
        INSERT INTO company_metadata
            (isin_code, ticker, groupe, first_seen_date, last_seen_date, total_trading_days, updated_at)
        VALUES %s
        ON CONFLICT (isin_code) DO UPDATE SET
            ticker          = EXCLUDED.ticker,
            groupe          = EXCLUDED.groupe,
            first_seen_date = LEAST(company_metadata.first_seen_date, EXCLUDED.first_seen_date),
            last_seen_date  = GREATEST(company_metadata.last_seen_date, EXCLUDED.last_seen_date),
            updated_at      = NOW()
    """

    rows = [
        (row.isin_code, row.ticker, TARGET_GROUP,
         row.first_seen_date, row.last_seen_date, 0, datetime.now())
        for row in companies.itertuples()
    ]

    with conn.cursor() as cur:
        execute_values(cur, sql, rows)
    conn.commit()
    print(f"  📋 company_metadata: upserted {len(rows)} companies")


def refresh_company_metadata_stats(conn, isin_codes: list[str]):
    """
    Recompute first/last dates and trading-day counts from daily_prices.
    This keeps metadata correct even when the loader is re-run.
    """
    if not isin_codes:
        return

    sql = """
        UPDATE company_metadata AS cm
        SET first_seen_date    = stats.first_seen_date,
            last_seen_date     = stats.last_seen_date,
            total_trading_days = stats.total_trading_days,
            updated_at         = NOW()
        FROM (
            SELECT
                isin_code,
                MIN(seance) AS first_seen_date,
                MAX(seance) AS last_seen_date,
                COUNT(*)    AS total_trading_days
            FROM daily_prices
            WHERE isin_code = ANY(%s)
            GROUP BY isin_code
        ) AS stats
        WHERE cm.isin_code = stats.isin_code
    """

    with conn.cursor() as cur:
        cur.execute(sql, (isin_codes,))
    conn.commit()


def insert_prices(conn, df: pd.DataFrame) -> int:
    """
    Bulk insert price rows.
    ON CONFLICT DO UPDATE = reruns can correct previously bad numeric values.
    """
    sql = """
        INSERT INTO daily_prices
            (seance, isin_code, ticker, ouverture, cloture,
             plus_bas, plus_haut, quantite_negociee, nb_transaction, capitaux)
        VALUES %s
        ON CONFLICT (seance, isin_code) DO UPDATE SET
            ticker            = EXCLUDED.ticker,
            ouverture         = EXCLUDED.ouverture,
            cloture           = EXCLUDED.cloture,
            plus_bas          = EXCLUDED.plus_bas,
            plus_haut         = EXCLUDED.plus_haut,
            quantite_negociee = EXCLUDED.quantite_negociee,
            nb_transaction    = EXCLUDED.nb_transaction,
            capitaux          = EXCLUDED.capitaux
    """

    rows = [
        (
            row.seance,
            row.isin_code,
            row.ticker,
            row.ouverture   if pd.notna(row.ouverture)          else None,
            row.cloture     if pd.notna(row.cloture)            else None,
            row.plus_bas    if pd.notna(row.plus_bas)           else None,
            row.plus_haut   if pd.notna(row.plus_haut)          else None,
            int(row.quantite_negociee) if pd.notna(row.quantite_negociee) else None,
            int(row.nb_transaction)    if pd.notna(row.nb_transaction)    else None,
            row.capitaux    if pd.notna(row.capitaux)           else None,
        )
        for row in df.itertuples()
    ]

    with conn.cursor() as cur:
        execute_values(cur, sql, rows, page_size=1000)
    conn.commit()

    return len(rows)

In [40]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 4: Main Orchestrator
# ═══════════════════════════════════════════════════════════════════════════

def main():
    print("=" * 60)
    print("  BVMT Unified Data Loader")
    print("=" * 60)

    # 1. Find all files
    csv_files = sorted(glob.glob(os.path.join(DATA_FOLDER, "*.csv")))
    txt_files = sorted(glob.glob(os.path.join(DATA_FOLDER, "*.txt")))
    all_files = csv_files + txt_files

    if not all_files:
        print(f"\n❌ No files found in: {DATA_FOLDER}")
        print("   Make sure your files are named histo_cotation_YYYY.csv / .txt")
        return

    print(f"\n📁 Found {len(all_files)} file(s) in: {DATA_FOLDER}")
    for f in all_files:
        print(f"   - {os.path.basename(f)}")

    # 2. Connect to DB
    print(f"\n🔌 Connecting to PostgreSQL ({DB_CONFIG['host']}:{DB_CONFIG['port']} / {DB_CONFIG['dbname']})...")
    try:
        conn = get_connection()
        print("   ✅ Connected!")
    except Exception as e:
        print(f"   ❌ Connection failed: {e}")
        print("   Check your .env file credentials and make sure PostgreSQL is running.")
        return

    # 3. Process each file
    total_rows_inserted = 0

    for filepath in all_files:
        print(f"\n{'─'*60}")
        print(f"Processing: {os.path.basename(filepath)}")
        print(f"{'─'*60}")

        try:
            # Parse
            df_raw = load_file(filepath)
            if df_raw.empty:
                continue

            # Clean & filter
            df = clean_and_filter(df_raw, filepath)
            if df.empty:
                continue

            # Upsert companies first
            upsert_company_metadata(conn, df)

            # Insert prices
            inserted = insert_prices(conn, df)
            total_rows_inserted += inserted
            refresh_company_metadata_stats(
                conn,
                df['isin_code'].dropna().astype(str).unique().tolist(),
            )
            print(f"  💾 Inserted {inserted:,} price rows into daily_prices")

        except Exception as e:
            print(f"  ❌ Error processing {os.path.basename(filepath)}: {e}")
            conn.rollback()
            continue

    # 4. Final summary
    print(f"\n{'='*60}")
    print("  LOAD COMPLETE")
    print(f"{'='*60}")
    print(f"  Total rows inserted: {total_rows_inserted:,}")

    # Quick verification query
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM daily_prices;")
        total_in_db = cur.fetchone()[0]

        cur.execute("SELECT COUNT(*) FROM company_metadata;")
        total_companies = cur.fetchone()[0]

        cur.execute("SELECT MIN(seance), MAX(seance) FROM daily_prices;")
        date_range = cur.fetchone()

    print(f"  Total rows in DB:    {total_in_db:,}")
    print(f"  Total companies:     {total_companies}")
    print(f"  Date range in DB:    {date_range[0]} → {date_range[1]}")
    print(f"{'='*60}\n")

    conn.close()


if __name__ == "__main__":
    main()




  BVMT Unified Data Loader

📁 Found 10 file(s) in: C:\Users\Negza\Desktop\projects\pfe\bvmt_project\data\prices
   - histo_cotation_2022.csv
   - histo_cotation_2023.csv
   - histo_cotation_2024.csv
   - histo_cotation_2025.csv
   - histo_cotation_2016.txt
   - histo_cotation_2017.txt
   - histo_cotation_2018.txt
   - histo_cotation_2019.txt
   - histo_cotation_2020.txt
   - histo_cotation_2021.txt

🔌 Connecting to PostgreSQL (localhost:5432 / bvmt_db)...
   ✅ Connected!

────────────────────────────────────────────────────────────
Processing: histo_cotation_2022.csv
────────────────────────────────────────────────────────────
  📂 Reading: histo_cotation_2022.csv
  ✅ 11,262 rows | 45 stocks | 2022-01-03 → 2022-12-30
  📋 company_metadata: upserted 45 companies
  💾 Inserted 11,262 price rows into daily_prices

────────────────────────────────────────────────────────────
Processing: histo_cotation_2023.csv
────────────────────────────────────────────────────────────
  📂 Reading: histo_cot

In [15]:
# ═══════════════════════════════════════════════════════════════════════════
# BH BANK Recovery Reload (single-stock backfill)
# ═══════════════════════════════════════════════════════════════════════════
# This cell reloads only BH/BH BANK rows from all source files and normalizes
# them to ticker='BH BANK' and isin_code='TN0001900604'.

BH_ISIN = 'TN0001900604'
BH_TICKER_ALIASES = {'BH', 'BH BANK'}

csv_files = sorted(glob.glob(os.path.join(DATA_FOLDER, '*.csv')))
txt_files = sorted(glob.glob(os.path.join(DATA_FOLDER, '*.txt')))
all_files = csv_files + txt_files

if not all_files:
    print(f"❌ No price files found in: {DATA_FOLDER}")
else:
    print(f"📁 BH reload will scan {len(all_files)} file(s)")

    conn = get_connection()
    total_rows_inserted = 0

    try:
        for filepath in all_files:
            filename = os.path.basename(filepath)
            print(f"\n{'─'*60}")
            print(f"Processing: {filename}")
            print(f"{'─'*60}")

            # Parse each file using the existing notebook parser functions.
            if filepath.lower().endswith('.csv'):
                df_raw = parse_csv(filepath)
            elif filepath.lower().endswith('.txt'):
                df_raw = parse_txt(filepath)
            else:
                print("  ⚠️ Unsupported extension; skipped")
                continue

            if df_raw.empty:
                print("  ⚠️ Empty source file")
                continue

            # Reuse the existing cleaning/standardization pipeline to build DB-ready columns.
            df_clean = clean_and_filter(df_raw, filepath)
            if df_clean.empty:
                print("  ⚠️ No valid market rows after clean/filter")
                continue

            # Keep only BH aliases, then normalize to a single ticker/ISIN identity.
            df_bh = df_clean[df_clean['ticker'].isin(BH_TICKER_ALIASES)].copy()
            if df_bh.empty:
                print("  ℹ️ No BH/BH BANK rows in this file")
                continue

            df_bh['ticker'] = 'BH BANK'
            df_bh['isin_code'] = BH_ISIN

            # Ensure company metadata exists/updates, then insert prices with built-in
            # ON CONFLICT handling from insert_prices (no duplicate rows on rerun).
            upsert_company_metadata(conn, df_bh)
            inserted = insert_prices(conn, df_bh)
            total_rows_inserted += inserted

            print(f"  ✅ BH rows prepared: {len(df_bh):,} | upserted: {inserted:,}")

        # Keep company metadata stats aligned with the final daily_prices state.
        refresh_company_metadata_stats(conn, [BH_ISIN])

        print(f"\n{'='*60}")
        print("BH BANK RELOAD COMPLETE")
        print(f"Total BH BANK rows inserted/upserted: {total_rows_inserted:,}")
        print(f"{'='*60}")

        # Verification query: confirm total days and date range now present in DB.
        with conn.cursor() as cur:
            cur.execute(
                """
                SELECT COUNT(*) AS days, MIN(seance) AS first_date, MAX(seance) AS last_date
                FROM daily_prices
                WHERE isin_code = %s AND ticker = 'BH BANK'
                """,
                (BH_ISIN,),
            )
            days, first_date, last_date = cur.fetchone()

        print(f"DB check (BH BANK): {days:,} days | {first_date} → {last_date}")

    except Exception as e:
        conn.rollback()
        print(f"❌ BH reload failed: {e}")
    finally:
        conn.close()
        print("🔒 Database connection closed")

📁 BH reload will scan 10 file(s)

────────────────────────────────────────────────────────────
Processing: histo_cotation_2022.csv
────────────────────────────────────────────────────────────
  ✅ 11,262 rows | 45 stocks | 2022-01-03 → 2022-12-30
  📋 company_metadata: upserted 1 companies
  ✅ BH rows prepared: 257 | upserted: 257

────────────────────────────────────────────────────────────
Processing: histo_cotation_2023.csv
────────────────────────────────────────────────────────────
  ✅ 10,044 rows | 41 stocks | 2023-01-02 → 2023-12-29
  📋 company_metadata: upserted 1 companies
  ✅ BH rows prepared: 252 | upserted: 252

────────────────────────────────────────────────────────────
Processing: histo_cotation_2024.csv
────────────────────────────────────────────────────────────
  ✅ 10,542 rows | 42 stocks | 2024-01-02 → 2024-12-31
  📋 company_metadata: upserted 1 companies
  ✅ BH rows prepared: 251 | upserted: 251

────────────────────────────────────────────────────────────
Processing: